In [1]:
import os

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI

from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper

/var/folders/yc/5l7k07cd6vvc8v6r03yz0x3h0000gn/T/ipykernel_38868/4090614986.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import GoogleSerperRun


In [2]:
sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

In [3]:
load_dotenv(override=True)

True

In [4]:
model = ChatOpenAI(
    model="gpt-5.4-mini"
)

In [5]:
@tool
def get_team_information(team: str) -> str:
    """
    Return information about a World Cup 2026 team.
    """

    teams = {

        "Brazil":
        "Brazil is one of the most successful World Cup nations with five titles.",

        "Argentina":
        "Argentina won the 2022 World Cup and enters 2026 as a major contender.",

        "France":
        "France won the 2018 World Cup and reached the 2022 final.",

        "USA":
        "The USA is one of the host nations for the 2026 World Cup."

    }

    return teams.get(
        team,
        "No information available."
    )

search = GoogleSerperRun(
    api_wrapper=GoogleSerperAPIWrapper()
)

In [ ]:
@tool
def send_notification(message: str) -> str:
    """
    Pretend to send a notification.
    """

    return f"Notification sent: {message}"

In [7]:
memory = InMemorySaver()

In [8]:
researcher = create_deep_agent(

    model=model,

    tools=[
        search,
        get_team_information,
        send_notification
    ],

    system_prompt=(
        "You are a World Cup 2026 research assistant."
        "Research teams and write reports."
    ),

    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_notification": True
            }
        )
    ],

    checkpointer=memory,

    backend=FilesystemBackend(
        root_dir=sandbox,
        virtual_mode=True
    ),
)

In [9]:
config = {
    "configurable": {
        "thread_id": "world-cup"
    }
}

In [10]:
resumed = researcher.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config=config
)

print(resumed["messages"][-1].content)

France: won the 2018 World Cup and reached the 2022 final.  
Brazil: one of the most successful World Cup nations with five titles.  
England: no information available.


In [11]:
import gradio as gr

def chat(user_input, history):

    result = researcher.invoke(
        {
            "messages": [
                HumanMessage(content=user_input)
            ]
        },
        config=config
    )

    print("\n===== TOOLS USED =====")

    for message in result["messages"]:
        if getattr(message, "tool_calls", None):
            for tool_call in message.tool_calls:
                print(f"Tool: {tool_call['name']}")
                print(f"Arguments: {tool_call['args']}\n")

    return result["messages"][-1].content

In [ ]:
gr.ChatInterface(
    fn=chat,
    title="⚽ World Cup 2026 Deep Research Agent",
    description="Ask questions or request World Cup reports."
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/Users/aradmishkal/projects/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)



===== TOOLS USED =====
Tool: get_team_information
Arguments: {'team': 'France'}

Tool: get_team_information
Arguments: {'team': 'Brazil'}

Tool: get_team_information
Arguments: {'team': 'England'}



/Users/aradmishkal/projects/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/aradmishkal/projects/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)



===== TOOLS USED =====
Tool: get_team_information
Arguments: {'team': 'France'}

Tool: get_team_information
Arguments: {'team': 'Brazil'}

Tool: get_team_information
Arguments: {'team': 'England'}

Tool: google_serper
Arguments: {'query': 'World Cup 2026 semi final teams current'}

Tool: get_team_information
Arguments: {'team': 'Argentina'}

Tool: get_team_information
Arguments: {'team': 'France'}

Tool: get_team_information
Arguments: {'team': 'Brazil'}

Tool: get_team_information
Arguments: {'team': 'England'}



...

/Users/aradmishkal/projects/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/aradmishkal/projects/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)



===== TOOLS USED =====
Tool: get_team_information
Arguments: {'team': 'France'}

Tool: get_team_information
Arguments: {'team': 'Brazil'}

Tool: get_team_information
Arguments: {'team': 'England'}

Tool: google_serper
Arguments: {'query': 'World Cup 2026 semi final teams current'}

Tool: get_team_information
Arguments: {'team': 'Argentina'}

Tool: get_team_information
Arguments: {'team': 'France'}

Tool: get_team_information
Arguments: {'team': 'Brazil'}

Tool: get_team_information
Arguments: {'team': 'England'}



/Users/aradmishkal/projects/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/aradmishkal/projects/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)



===== TOOLS USED =====
Tool: get_team_information
Arguments: {'team': 'France'}

Tool: get_team_information
Arguments: {'team': 'Brazil'}

Tool: get_team_information
Arguments: {'team': 'England'}

Tool: google_serper
Arguments: {'query': 'World Cup 2026 semi final teams current'}

Tool: get_team_information
Arguments: {'team': 'Argentina'}

Tool: get_team_information
Arguments: {'team': 'France'}

Tool: get_team_information
Arguments: {'team': 'Brazil'}

Tool: get_team_information
Arguments: {'team': 'England'}

Tool: send_notification
Arguments: {'message': 'Notification requested by user.'}



/Users/aradmishkal/projects/agents2/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
